# Compare all cryptocurrencies
This document predicts which cryptocurrency is going to have the largest change in price

In [1]:
! rm *.csv

from google.colab import files
uploaded = files.upload()
%ls

rm: cannot remove '*.csv': No such file or directory


Saving current_sentiment_2h.csv to current_sentiment_2h.csv
Saving historical_sentiment_2h.csv to historical_sentiment_2h.csv
Saving price_2h.csv to price_2h.csv
current_sentiment_2h.csv     price_2h.csv
historical_sentiment_2h.csv  sample_data/


## Setup

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder
import random

In [3]:
from os import listdir
from os.path import isfile

Importing the collected data

In [4]:
cryptos = ["biance-coin", "bitcoin-cash", "bitcoin", "cardano", "chainlink", "ethereum", "litecoin", "ripple", "stellar"]

In [5]:
mm = MinMaxScaler()
ss = StandardScaler()

le = LabelEncoder()

## Helper Methods

In [6]:
def plot_time_series(predicted, true, n_training, filename):
  """
  Plot the time series
  """
  plt.figure(figsize=(8,6)) #plotting
  plt.axvline(x=n_training, color="#ffd166", linestyle='-') #size of the training set

  plt.plot(predicted, label='Predicted Price', color="#118ab2") #predicted plot
  plt.plot(true, label='True Price', color="#06d6a0") #actual plot

  plt.title('Time-Series Prediction', fontsize=16)
  plt.xlabel('Time', fontsize=14)
  plt.ylabel('Price', fontsize=14)

  plt.xlim(0)
  plt.legend()
  plt.show()
  #plt.savefig(filename) 

In [7]:
def classify(predicted_best, true_best):
    df = pd.DataFrame([predicted_best, true_best], columns=["pred","true"])

    total_correct = (predicted_best == true_best).sum()
    return totalcorrect/len(predict)

## LSTM Model


In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using {} device'.format(device))

Using cuda device


### Model Definition

LSTM Class used: https://pytorch.org/docs/master/generated/torch.nn.LSTM.html#torch.nn.LSTM

Some tutorials: https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html



In [9]:
class LSTMCustom(nn.Module):
    def __init__(self, num_classes, input_size, hidden_size, num_layers, seq_length):
        super(LSTMCustom, self).__init__()
        self.num_classes = num_classes #number of classes
        self.num_layers = num_layers #number of layers
        self.input_size = input_size #input size
        self.hidden_size = hidden_size #hidden state
        self.seq_length = seq_length #sequence length

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout = 0.05
            )
        #self.fc_1 =  nn.Linear(hidden_size, 128) #fully connected 1
        #self.fc = nn.Linear(128, num_classes) #fully connected last layer

        #self.relu = nn.ReLU()

        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self,x):
        h_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #hidden state
        c_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #internal state
        # Propagate input through LSTM
        output, (hn, cn) = self.lstm(x, (h_0, c_0)) #lstm with input, hidden, and internal state

        out = self.fc(output[:,-1,:])
        return out

### Model Parameters

In [10]:
num_epochs = 500
learning_rate = 0.02

hidden_size = 32 #number of features in hidden state
num_layers = 12 #number of stacked lstm layers

look_back = 12

# probably dont change
num_classes = len(cryptos)

In [11]:
criterion = torch.nn.MSELoss()  # mean-squared error for regression

### Methods

In [12]:
def train_test_split_tensor(x, y):
  """
  Custom train/test splitting
  TODO: maybe shorten code by using sklearn.preprocessing.train_test_split
  """
  cutoff = round(x.shape[0] * 0.8)

  # split into train and test
  x_train = x[:cutoff, :]
  x_test = x[cutoff:,:]
  y_train = y[:cutoff, :]
  y_test = y[cutoff:, :]
  
  return x_train, x_test, y_train, y_test, cutoff

In [13]:
def convert_2d(x_ss):
  x_2d = []
  for index in range(look_back, len(x_ss)):
    x_2d.append(np.array(x_ss[index-look_back:index+1]))
  x_2d = np.array(x_2d)
  return x_2d

In [14]:
def get_x_y(df):
  # split into x and y
  y = le.fit_transform(df["best_crypto"])
  y = nn.functional.one_hot(torch.tensor(y, dtype=torch.int64), len(cryptos))

  x = df.drop(["best_crypto", "best_diff"],axis=1)
  x_ss = ss.fit_transform(x)
  # x_ss = x.values

  x_2d = torch.tensor(x_ss)
  if sentiment:
    x_2d = convert_2d(x_ss)
    x_2d = Variable(torch.Tensor(x_2d))
    y = y[:len(y)-look_back]
  else:
    x_2d = Variable(torch.reshape(x_2d, (x_2d.shape[0], 1, x_2d.shape[1])))
  
  return x_2d, y

In [15]:
def train(x_train, y_train, lstm):
  """
  Train the lstm
  """

  for epoch in range(num_epochs):
    outputs = lstm.forward(x_train) #forward pass
    optimizer.zero_grad() #caluclate the gradient, manually setting to 0
  
    # obtain the loss function
    loss = criterion(outputs, y_train)
  
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
      print("Epoch: %d, loss: %1.5f" % (epoch, loss.item())) 

  return lstm, cutoff

## Execution

### Historical Price and Sentiment

In [16]:
sentiment = True

In [17]:
df = pd.read_csv("historical_sentiment_2h.csv")

x_2d, y = get_x_y(df)
x_train, x_test, y_train, y_test, cutoff = train_test_split_tensor(x_2d, y)


In [18]:
print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)


torch.Size([3493, 13, 549]) torch.Size([873, 13, 549])
torch.Size([3493, 9]) torch.Size([873, 9])


In [19]:
input_size = x_train.size(2)
# input_size = 1

In [20]:
lstm = LSTMCustom(num_classes, input_size, hidden_size, num_layers, df.shape[1]) 
optimizer = torch.optim.Adam(lstm.parameters(), lr=learning_rate) 

lstm, cutoff = train(x_train.float(), y_train.float(), lstm)

Epoch: 0, loss: 0.14182
Epoch: 50, loss: 0.06800
Epoch: 100, loss: 0.05213
Epoch: 150, loss: 0.04919
Epoch: 200, loss: 0.04247
Epoch: 250, loss: 0.04208
Epoch: 300, loss: 0.03710
Epoch: 350, loss: 0.03777
Epoch: 400, loss: 0.04012
Epoch: 450, loss: 0.03682


In [21]:
torch.save(lstm.state_dict(), "predict_crypto_sent.model")

#### Results

In [22]:
preds = lstm(x_2d.float()).argmax(axis=1)
y_true = y.argmax(axis=1)

In [23]:
train_pred = preds[:cutoff]
train_y = y_true[:cutoff]

train_acc = (train_pred == train_y).sum()/len(preds)
train_acc

tensor(0.6093)

In [24]:
test_pred = preds[cutoff:]
test_y = y_true[cutoff:]

test_acc = (test_pred == test_y).sum()/len(preds)
test_acc

tensor(0.0323)

In [25]:
print("input size: ", input_size)
print("epochs: ", num_epochs)
print("learning rate: ", learning_rate)
print("hidden size: ", hidden_size)
print("look back: ", look_back)

input size:  549
epochs:  500
learning rate:  0.02
hidden size:  32
look back:  12


In [26]:
pd.DataFrame([[train_acc, test_acc]], columns=["train accuracy", "test accuracy"])

,train accuracy,test accuracy
0,tensor(0.6093),tensor(0.0323)


In [27]:
torch.save(lstm.state_dict(), "predict_crypto_hist_sent.model")

### Current Sentiment Historical Price

In [28]:
sentiment = True

In [29]:
df = pd.read_csv("current_sentiment_2h.csv")

x_2d, y = get_x_y(df)
x_train, x_test, y_train, y_test, cutoff = train_test_split_tensor(x_2d, y)


In [30]:
print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)


torch.Size([3493, 13, 225]) torch.Size([873, 13, 225])
torch.Size([3493, 9]) torch.Size([873, 9])


In [31]:
input_size = x_train.size(2)
# input_size = 1

In [32]:
lstm = LSTMCustom(num_classes, input_size, hidden_size, num_layers, df.shape[1]) 
optimizer = torch.optim.Adam(lstm.parameters(), lr=learning_rate) 

lstm, cutoff = train(x_train.float(), y_train.float(), lstm)

Epoch: 0, loss: 0.10502
Epoch: 50, loss: 0.07418
Epoch: 100, loss: 0.07418
Epoch: 150, loss: 0.07417
Epoch: 200, loss: 0.07149
Epoch: 250, loss: 0.07312
Epoch: 300, loss: 0.07154
Epoch: 350, loss: 0.07141
Epoch: 400, loss: 0.07040
Epoch: 450, loss: 0.07108


In [33]:
torch.save(lstm.state_dict(), "predict_crypto_sent.model")

#### Results

In [34]:
preds = lstm(x_2d.float()).argmax(axis=1)
y_true = y.argmax(axis=1)

In [35]:
train_pred = preds[:cutoff]
train_y = y_true[:cutoff]

train_acc = (train_pred == train_y).sum()/len(preds)
train_acc

tensor(0.4579)

In [36]:
test_pred = preds[cutoff:]
test_y = y_true[cutoff:]

test_acc = (test_pred == test_y).sum()/len(preds)
test_acc

tensor(0.1083)

In [37]:
print("input size: ", input_size)
print("epochs: ", num_epochs)
print("learning rate: ", learning_rate)
print("hidden size: ", hidden_size)
print("look back: ", look_back)

input size:  225
epochs:  500
learning rate:  0.02
hidden size:  32
look back:  12


In [38]:
pd.DataFrame([[train_acc, test_acc]], columns=["train accuracy", "test accuracy"])

,train accuracy,test accuracy
0,tensor(0.4579),tensor(0.1083)


In [39]:
torch.save(lstm.state_dict(), "predict_crypto_current_sent.model")

### Price Only


In [40]:
sentiment = False

In [41]:
df = pd.read_csv("price_2h.csv")

x_2d, y = get_x_y(df)
x_train, x_test, y_train, y_test, cutoff = train_test_split_tensor(x_2d, y)


In [42]:
print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)


torch.Size([3502, 1, 252]) torch.Size([876, 1, 252])
torch.Size([3502, 9]) torch.Size([876, 9])


In [43]:
input_size = x_train.size(2)
input_size

252

In [44]:
lstm = LSTMCustom(num_classes, input_size, hidden_size, num_layers, df.shape[1]) 
optimizer = torch.optim.Adam(lstm.parameters(), lr=learning_rate) 

lstm, cutoff = train(x_train.float(), y_train.float(), lstm)

Epoch: 0, loss: 0.11570
Epoch: 50, loss: 0.07406
Epoch: 100, loss: 0.07405
Epoch: 150, loss: 0.07404
Epoch: 200, loss: 0.07406
Epoch: 250, loss: 0.07379
Epoch: 300, loss: 0.07372
Epoch: 350, loss: 0.07377
Epoch: 400, loss: 0.07368
Epoch: 450, loss: 0.07361


#### Results


In [45]:
preds = lstm(x_2d.float()).argmax(axis=1)
y_true = y.argmax(axis=1)

In [46]:
train_pred = preds[:cutoff]
train_y = y_true[:cutoff]

train_acc = (train_pred == train_y).sum()/len(preds)
train_acc

tensor(0.4399)

In [47]:
test_pred = preds[cutoff:]
test_y = y_true[cutoff:]

test_acc = (test_pred == test_y).sum()/len(preds)
test_acc

tensor(0.1300)

In [48]:
print("input size: ", input_size)
print("epochs: ", num_epochs)
print("learning rate: ", learning_rate)
print("hidden size: ", hidden_size)
print("look back: ", look_back)

input size:  252
epochs:  500
learning rate:  0.02
hidden size:  32
look back:  12


In [49]:
pd.DataFrame([[train_acc, test_acc]], columns=["train accuracy", "test accuracy"])

,train accuracy,test accuracy
0,tensor(0.4399),tensor(0.1300)


### Save Model

In [50]:
torch.save(lstm.state_dict(), "predict_crypto_price.model")

In [51]:
# files.download("predict_crypto_sent.model")
# files.download("predict_crypto_price.model")